# Day 2 — QLoRA Fine-Tuning (TinyLlama 1.1B)

Fine-tune **TinyLlama/TinyLlama-1.1B-Chat-v1.0** using **QLoRA** (4-bit NF4 + LoRA) on the Day 1 instruction dataset.

| Param | Value |
|---|---|
| LoRA rank (r) | 16 |
| LoRA alpha | 32 |
| Learning rate | 2e-4 |
| Batch size | 4 |
| Epochs | 3 |
| Quantization | 4-bit NF4 |

> **Runtime**: Set to **GPU → T4** before running.

In [ ]:
# Check GPU and install dependencies
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!pip install -q transformers datasets peft trl accelerate bitsandbytes

In [ ]:
import torch, json, os
import matplotlib.pyplot as plt
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


In [ ]:
# Load model with 4-bit quantization
MODEL_ID = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
)
model.config.use_cache = False

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

print(f'Model loaded: {MODEL_ID}')
print(f'Memory footprint: {model.get_memory_footprint() / 1e9:.2f} GB')

In [ ]:
# Apply LoRA adapters
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Format dataset into TinyLlama chat template
def format_sample(example):
    instruction = example['instruction']
    inp = example['input']
    output = example['output']
    user_msg = f'{instruction}\n{inp}'.strip() if inp else instruction
    return {'text': f'<|user|>\n{user_msg}</s>\n<|assistant|>\n{output}</s>'}

train_dataset = train_dataset.map(format_sample)
val_dataset = val_dataset.map(format_sample)

print('Formatted sample:')
print(train_dataset[0]['text'][:500])

In [ ]:
# Training configuration and run
sft_config = SFTConfig(
    output_dir='/content/qlora-output',
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    weight_decay=0.01,
    logging_steps=10,
    save_strategy='epoch',
    eval_strategy='epoch',
    fp16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    max_seq_length=512,
    dataset_text_field='text',
    packing=False,
    optim='paged_adamw_32bit',
    warmup_ratio=0.03,
    lr_scheduler_type='cosine',
    report_to='none',
    save_total_limit=2,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    args=sft_config,
)

print(f'Training steps per epoch: ~{len(train_dataset) // (4 * 4)}')
train_result = trainer.train()
print(train_result.metrics)

In [ ]:
# Plot training and evaluation loss
log_history = trainer.state.log_history
train_loss = [(x['step'], x['loss']) for x in log_history if 'loss' in x]
eval_loss  = [(x['step'], x['eval_loss']) for x in log_history if 'eval_loss' in x]

fig, ax = plt.subplots(figsize=(10, 5))
if train_loss:
    steps, losses = zip(*train_loss)
    ax.plot(steps, losses, label='Train Loss', marker='o', markersize=3)
if eval_loss:
    steps, losses = zip(*eval_loss)
    ax.plot(steps, losses, label='Eval Loss', marker='s', markersize=5)
ax.set_xlabel('Step')
ax.set_ylabel('Loss')
ax.set_title('QLoRA Fine-Tuning — Loss Curve')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/content/training_loss.png', dpi=150)
plt.show()

In [ ]:
# Save adapter weights
ADAPTER_DIR = '/content/adapters'
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print('Saved adapter files:')
for f in sorted(os.listdir(ADAPTER_DIR)):
    size = os.path.getsize(os.path.join(ADAPTER_DIR, f))
    print(f'  {f} — {size / 1024:.1f} KB')

In [ ]:
# Test inference with the fine-tuned model
model.eval()
prompts = [
    'What is gradient descent in machine learning?',
    'Explain the difference between SQL and NoSQL databases.',
    'What are the benefits of using Docker containers?',
]

for p in prompts:
    inputs = tokenizer(
        f'<|user|>\n{p}</s>\n<|assistant|>\n',
        return_tensors='pt',
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.1,
        )
    response = tokenizer.decode(out[0], skip_special_tokens=False)
    answer = response.split('<|assistant|>')[-1].replace('</s>', '').strip()
    print(f'Q: {p}')
    print(f'A: {answer}')
    print('=' * 60)

In [ ]:
# Generate TRAINING-REPORT.md with actual metrics
eval_results = trainer.evaluate()
metrics = train_result.metrics
total_p = sum(p.numel() for p in model.parameters())
train_p = sum(p.numel() for p in model.parameters() if p.requires_grad)

report = f'''# TRAINING-REPORT.md — Week 8 Day 2

## Model
- **Base**: TinyLlama/TinyLlama-1.1B-Chat-v1.0
- **Method**: QLoRA (4-bit NF4 + LoRA)
- **Framework**: transformers + peft + trl + bitsandbytes

## LoRA Configuration
| Parameter | Value |
|-----------|-------|
| Rank (r) | 16 |
| Alpha | 32 |
| Dropout | 0.05 |
| Target Modules | q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj |
| Task Type | CAUSAL_LM |

## Quantization
| Parameter | Value |
|-----------|-------|
| Precision | 4-bit |
| Quant Type | NF4 |
| Compute Dtype | float16 |
| Double Quant | Yes |

## Training Configuration
| Parameter | Value |
|-----------|-------|
| Epochs | 3 |
| Batch Size | 4 |
| Gradient Accumulation | 4 |
| Effective Batch Size | 16 |
| Learning Rate | 2e-4 |
| Scheduler | Cosine |
| Warmup Ratio | 0.03 |
| Optimizer | paged_adamw_32bit |
| Max Seq Length | 512 |
| FP16 | Yes |
| Gradient Checkpointing | Yes |

## Dataset
| Split | Samples |
|-------|---------|
| Train | {len(train_dataset)} |
| Val | {len(val_dataset)} |

## Trainable Parameters
- **Total**: {total_p:,}
- **Trainable**: {train_p:,}
- **Trainable %**: {100 * train_p / total_p:.2f}%

## Results
| Metric | Value |
|--------|-------|
| Final Train Loss | {metrics.get('train_loss', 0):.4f} |
| Eval Loss | {eval_results.get('eval_loss', 0):.4f} |
| Train Runtime (s) | {metrics.get('train_runtime', 0):.1f} |
| Samples/sec | {metrics.get('train_samples_per_second', 0):.2f} |

## Loss Curve
![Training Loss](training_loss.png)

## Adapter Output
- adapter_model.safetensors
- adapter_config.json
'''

with open('/content/TRAINING-REPORT.md', 'w') as f:
    f.write(report)
print(report)

In [ ]:
# Download all artifacts
import shutil

shutil.make_archive('/content/adapters', 'zip', ADAPTER_DIR)
print(f'Adapter zip: {os.path.getsize("/content/adapters.zip") / 1e6:.1f} MB')

from google.colab import files
files.download('/content/adapters.zip')
files.download('/content/TRAINING-REPORT.md')
files.download('/content/training_loss.png')

# Day 2 — QLoRA Fine-Tuning with LoRA Adapters

Fine-tune **Qwen/Qwen2.5-1.5B-Instruct** using QLoRA on the custom instruction dataset from Day 1.

**Config:** r=16, alpha=32, lr=2e-4, batch=4, epochs=3, 4-bit NF4  
**Goal:** Trainable params ~1%, loss decreasing, adapter weights saved.

## 1. Install and Import Dependencies

Install required libraries for QLoRA fine-tuning. If running on Colab, these will install into the runtime.

In [ ]:
# Pin exact versions to guarantee API compatibility
!pip install -q \
  "torch==2.5.1" \
  "transformers==4.46.3" \
  "peft==0.14.0" \
  "bitsandbytes==0.44.1" \
  "trl==0.12.2" \
  "datasets==3.1.0" \
  "accelerate==1.1.1" \
  "matplotlib==3.9.3"

In [ ]:
import os
from google.colab import files

DATA_PATH   = "/content/train.jsonl"
ADAPTER_DIR = "/content/adapters"
REPORT_PATH = "/content/TRAINING-REPORT.md"

os.makedirs(ADAPTER_DIR, exist_ok=True)

print("Upload train.jsonl:")
files.upload()

In [ ]:
import os, torch, matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, PeftModel, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

import trl, transformers, peft
print(f"trl={trl.__version__}  transformers={transformers.__version__}  peft={peft.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name}  VRAM: {props.total_memory/1e9:.1f} GB")

## 2. Load Dataset from JSONL

Load the instruction-tuning dataset prepared in Day 1. Each sample has `instruction`, `input`, and `output` fields.

In [ ]:
dataset = load_dataset("json", data_files=DATA_PATH, split="train")
print(f"{len(dataset)} samples | columns: {dataset.column_names}")
print(dataset[0])

## 3. Load Base Model with 4-bit Quantization (BitsAndBytes)

Load **Qwen2.5-1.5B-Instruct** with NF4 4-bit quantization. `double_quant=True` saves an extra ~0.4 bits/param.

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)
model = prepare_model_for_kbit_training(model)
model.gradient_checkpointing_enable()
print(f"Loaded {MODEL_NAME}")

## 4. Load Tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f"Vocab: {tokenizer.vocab_size} | Pad: '{tokenizer.pad_token}'")

## 5. Apply LoRA Configuration (r=16, alpha=32, dropout=0.05)

LoRA injects trainable rank-decomposition matrices into attention layers. Only adapters are trained; base weights stay frozen.

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 6. Preprocess and Tokenize Dataset

Format each sample into an **Alpaca-style** prompt template that the model learns to complete. The SFTTrainer handles tokenization internally via the formatting function.

In [ ]:
def format_instruction(sample):
    if sample["input"] and sample["input"].strip():
        return (f"### Instruction:\n{sample['instruction']}\n\n"
                f"### Input:\n{sample['input']}\n\n"
                f"### Response:\n{sample['output']}")
    return (f"### Instruction:\n{sample['instruction']}\n\n"
            f"### Response:\n{sample['output']}")

print(format_instruction(dataset[0])[:500])

## 7. Configure Training Arguments (lr=2e-4, batch=4, epochs=3)

In [ ]:
# trl==0.12.2: SFTConfig owns max_seq_length and packing
training_args = SFTConfig(
    output_dir=ADAPTER_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=50,
    fp16=True,
    gradient_checkpointing=True,
    logging_steps=10,
    save_strategy="epoch",
    optim="paged_adamw_32bit",
    report_to="none",
    seed=42,
    max_seq_length=512,
    packing=False,
)

## 8. Train with SFTTrainer

In [ ]:
# trl==0.12.2: SFTTrainer uses processing_class (not tokenizer)
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    processing_class=tokenizer,
    args=training_args,
    formatting_func=format_instruction,
)

train_result = trainer.train()
print(f"Loss: {train_result.training_loss:.4f} | "
      f"Runtime: {train_result.metrics['train_runtime']:.1f}s")

## 9. Plot Training Loss Curve

Visualize the training loss over steps to confirm it is decreasing (model is learning).

In [ ]:
log_history = trainer.state.log_history
train_losses = [(e["step"], e["loss"]) for e in log_history if "loss" in e]

steps, losses = zip(*train_losses)
plt.figure(figsize=(10, 5))
plt.plot(steps, losses, marker="o", markersize=3, linewidth=1.5, color="steelblue")
plt.title("Training Loss Curve (QLoRA)")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("/content/training_loss_curve.png", dpi=150)
plt.show()
print(f"Initial: {losses[0]:.4f} → Final: {losses[-1]:.4f}")

## 10. Verify Trainable Parameters (~1%)

Confirm that only the LoRA adapter parameters are trainable and they represent approximately 1% or less of the total model parameters.

In [ ]:
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params     = sum(p.numel() for p in model.parameters())
trainable_pct    = 100 * trainable_params / total_params
print(f"Trainable: {trainable_params:,} / {total_params:,} ({trainable_pct:.2f}%)")
assert trainable_pct < 5.0, f"Expected <5%, got {trainable_pct:.2f}%"

## 11. Save Adapter Weights

Save only the LoRA adapter weights (a few MB) — not the full model (several GB). The adapter can be loaded on top of any copy of the base model later.

In [ ]:
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"Saved to {ADAPTER_DIR}")
print("Files:", os.listdir(ADAPTER_DIR))

## 12. Test Inference with Fine-tuned Adapter

In [ ]:
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map="auto"
)
ft_model     = PeftModel.from_pretrained(base_model, ADAPTER_DIR).eval()
ft_tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)

def generate(prompt, max_new_tokens=150):
    inputs = ft_tokenizer(prompt, return_tensors="pt").to(ft_model.device)
    with torch.no_grad():
        out = ft_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=ft_tokenizer.pad_token_id,
        )
    return ft_tokenizer.decode(out[0], skip_special_tokens=True)

for i, p in enumerate([
    "### Instruction:\nWhat is GitHub Actions?\n\n### Response:\n",
    "### Instruction:\nWhat are the pros and cons of denormalizing data sets?\n\n### Response:\n",
], 1):
    print(f"--- Test {i} ---\n{generate(p)}\n")

## 13. Generate Training Report Summary

Compile all training metadata into a structured report and save as `TRAINING-REPORT.md`.

In [ ]:
final_loss    = train_result.training_loss
runtime       = train_result.metrics["train_runtime"]
samples_per_s = train_result.metrics.get("train_samples_per_second", 0)

report = f"""# TRAINING-REPORT — Week 8 Day 2

## Base Model
{MODEL_NAME} (1.5B parameters). Instruction-tuned, Colab T4-compatible.

## Quantization
4-bit NF4 (BitsAndBytes), double quantization enabled, compute dtype FP16.

## LoRA Config
r=16, alpha=32, dropout=0.05, target=q_proj/v_proj, bias=none.

## Training Config
lr=2e-4, cosine decay, warmup_steps=50, batch=4, epochs=3, {len(dataset)} samples.
FP16 + gradient checkpointing. Optimizer: paged_adamw_32bit. max_seq_length=512.

## Parameter Efficiency
Total: {total_params:,} | Trainable: {trainable_params:,} ({trainable_pct:.2f}%)

## Results
Loss: {final_loss:.4f} | Runtime: {runtime:.1f}s | Throughput: {samples_per_s:.1f} samples/s

## Artifacts
Adapters: {ADAPTER_DIR}
Loss curve: /content/training_loss_curve.png
"""

with open(REPORT_PATH, "w") as f:
    f.write(report)
print(report)